# RAG Layer — Unit Tests

Step-by-step testing of `openapi_generator.rag` (qdrant_factory + retriever).
Run each cell independently to inspect intermediate outputs and logs.

**Pre-requisites (only for the connectivity / retrieval cells):**
1. Qdrant container running on the host:port from `.env` (default `localhost:6333`).
2. Collection from `QDRANT_COLLECTION` (default `3gpp_rel18_28`) already indexed.

In [1]:
# Step 1 — Imports and setup
# Run this cell first.

import socket

from openapi_generator.config import get_logger
from openapi_generator.config.settings import (
    QDRANT_COLLECTION,
    QDRANT_HOST,
    QDRANT_PORT,
)
from openapi_generator.rag import retriever

logger = get_logger(__name__)
logger.info(f"Qdrant target  : {QDRANT_HOST}:{QDRANT_PORT}")
logger.info(f"Collection     : {QDRANT_COLLECTION}")

/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_generator/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-26 21:09:57 [INFO] __main__: Qdrant target  : localhost:6333
2026-05-26 21:09:57 [INFO] __main__: Collection     : 3gpp_rel18_28


In [2]:
# Step 2 — TCP reachability of QDRANT_HOST:QDRANT_PORT

try:
    with socket.create_connection((QDRANT_HOST, QDRANT_PORT), timeout=5):
        pass
    logger.info(f"Qdrant reachable at {QDRANT_HOST}:{QDRANT_PORT}")
    qdrant_up = True
except OSError as e:
    logger.warning(f"Qdrant NOT reachable: {e}. Skip cells below.")
    qdrant_up = False

2026-05-26 21:09:57 [INFO] __main__: Qdrant reachable at localhost:6333


In [3]:
# Step 3 — Retrieval over the live collection
# Simple sanity check: retrieve a few chunks and log a preview.

assert qdrant_up, "Qdrant is not reachable — see Step 2."

chunks = retriever.get_relevant_chunks(query="alarm notification", k=3)
logger.info(f"Retrieved {len(chunks)} chunk(s)")
for i, c in enumerate(chunks):
    preview = c[:200].replace("\n", " ")
    logger.info(f"  Chunk {i + 1}: {preview!r}")

2026-05-26 21:09:58 [INFO] openapi_generator.rag.qdrant_factory: Connecting to Qdrant at localhost:6333
/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_generator/.venv/lib/python3.13/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.16.3. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(
2026-05-26 21:09:58 [INFO] openapi_generator.config.hardware: Embedding device: cuda
2026-05-26 21:09:58 [INFO] openapi_generator.rag.qdrant_factory: Loading embeddings: sentence-transformers/all-MiniLM-L6-v2 on cuda
2026-05-26 21:10:03 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-26 21:10:03 [INFO] sentence_tr

In [4]:
# Step 4 — Retrieval over the OpenAPI 3.0 reference collection
# The collection is populated externally (e.g. by openapi_rulesbank).

from openapi_generator.tools.rag_tools import search_openapi_reference

assert qdrant_up, "Qdrant is not reachable — see Step 2."

result = search_openapi_reference(query="path parameter required schema", k=3)
logger.info(f"Retrieved {len(result)} chars of OpenAPI reference context")
if result:
    preview = result[:400].replace("\n", " ")
    logger.info(f"  Preview: {preview!r}")

/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_generator/.venv/lib/python3.13/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.16.3. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(
2026-05-26 21:10:09 [INFO] sentence_transformers.base.model: Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5201.41it/s]
2026-05-26 21:10:15 [INFO] __main__: Retrieved 2013 chars of OpenAPI reference context
2026-05-26 21:10:15 [INFO] __main__:   Preview: '| <a name="parameter-schema"></a>schema | [Schema Object](#schema-object) | The schema defining the type used for the parameter. |  ---  #### Options for Mapping Values to Schemas  The value of the property name